# Phase 5 — Practical footprint masks: one annotated frame, SAM 2, ROSE
Replaces the oracle mask with the realistic protocol: four clicks on the middle
frame, SAM 2 propagates the regions across the orbit, ROSE edits.
Runtime: GPU; no restarts needed.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Orbit frames -> zero-indexed jpgs (SAM 2's expected video format)
import os, glob, shutil
DRIVE = '/content/drive/MyDrive/light-footprint-removal'
if not os.path.isdir('/content/orbit_frames'):
    shutil.copytree(f'{DRIVE}/rose_inputs/orbit_frames', '/content/orbit_frames')
frame_paths = sorted(glob.glob('/content/orbit_frames/*.jpg'))
assert len(frame_paths) == 81
os.makedirs('/content/sam2_frames', exist_ok=True)
for i, p in enumerate(frame_paths):
    shutil.copy(p, f'/content/sam2_frames/{i:05d}.jpg')

In [ ]:
!pip -q install --force-reinstall "huggingface_hub==0.36.2"
!pip -q install "git+https://github.com/facebookresearch/sam2.git"
import torch
from sam2.sam2_video_predictor import SAM2VideoPredictor
predictor = SAM2VideoPredictor.from_pretrained('facebook/sam2.1-hiera-large')

In [ ]:
# THE ANNOTATION — the only manual step of the practical pipeline.
# Adjust (x, y) until each X sits on its target, re-running to check.
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

ANCHOR = 40                                  # middle of the arc (file 0041)
CLICKS = {1: ('sphere',           [(410, 260)]),
          2: ('mirror image',     [(560, 185)]),
          3: ('shadow',           [(600, 310)]),
          4: ('floor reflection', [(415, 395)])}

plt.figure(figsize=(12, 7))
plt.imshow(Image.open(f'/content/sam2_frames/{ANCHOR:05d}.jpg'))
for oid, (name, pts) in CLICKS.items():
    for x, y in pts:
        plt.scatter([x], [y], s=120, marker='x')
        plt.annotate(name, (x, y), color='yellow', xytext=(x + 12, y - 10))
plt.axis('off'); plt.show()

In [ ]:
# Propagate all four regions through the orbit (both directions from the anchor)
state = predictor.init_state(video_path='/content/sam2_frames')
for oid, (name, pts) in CLICKS.items():
    predictor.add_new_points_or_box(
        inference_state=state, frame_idx=ANCHOR, obj_id=oid,
        points=np.array(pts, np.float32), labels=np.ones(len(pts), np.int32))

N, W, H = 81, 832, 480
union = np.zeros((N, H, W), bool)
with torch.inference_mode():
    for reverse in (False, True):
        for fidx, obj_ids, logits in predictor.propagate_in_video(state, reverse=reverse):
            for i, _ in enumerate(obj_ids):
                union[fidx] |= (logits[i, 0] > 0).cpu().numpy()
print('mean coverage:', round(float(union.mean()), 3))

In [ ]:
# Union + dilation -> mask video; verify coverage at start / middle / end
from PIL import ImageFilter
import imageio
masks = []
for i in range(N):
    m = Image.fromarray((union[i] * 255).astype('uint8')).filter(ImageFilter.MaxFilter(15))
    masks.append(m.point(lambda v: 255 if v > 127 else 0))

fig, ax = plt.subplots(1, 3, figsize=(18, 5))
for a, idx in zip(ax, (0, 40, 80)):
    base = np.asarray(Image.open(f'/content/sam2_frames/{idx:05d}.jpg')).copy()
    sel = np.array(masks[idx]) > 127
    base[sel] = (0.5 * base[sel] + [127, 0, 0]).astype('uint8')
    a.imshow(base); a.set_title(f'frame {idx + 1}'); a.axis('off')
plt.show()

imageio.mimsave('/content/mask_sam2.mp4', [np.array(m.convert('RGB')) for m in masks], fps=16)
shutil.copy('/content/mask_sam2.mp4', f'{DRIVE}/rose_inputs/mask_sam2.mp4')

In [ ]:
# ROSE setup (idempotent; identical fixes to phase4b)
import os, shutil
%cd /content
if not os.path.isdir('/content/ROSE'):
    !git clone https://github.com/Kunbyte-AI/ROSE.git
%cd /content/ROSE
if not os.path.exists('/content/ROSE/.deps_done'):
    !pip -q install -r requirements.txt
    !pip -q install "transformers==4.46.2" "tokenizers>=0.20,<0.21"
    !pip -q install --force-reinstall "huggingface_hub==0.36.2"
    open('/content/ROSE/.deps_done', 'w').write('1')
from huggingface_hub import snapshot_download
if not os.path.isdir('/content/ROSE/models/Wan2.1-Fun-1.3B-InP'):
    snapshot_download('alibaba-pai/Wan2.1-Fun-1.3B-InP',
                      local_dir='/content/ROSE/models/Wan2.1-Fun-1.3B-InP')
if not os.path.exists('/content/ROSE/weights/transformer/config.json'):
    snapshot_download('Kunbyte/ROSE', local_dir='/content/ROSE/weights')
    os.makedirs('/content/ROSE/weights/transformer', exist_ok=True)
    for f in ('config.json', 'diffusion_pytorch_model.safetensors'):
        if os.path.exists(f'/content/ROSE/weights/{f}'):
            shutil.move(f'/content/ROSE/weights/{f}',
                        f'/content/ROSE/weights/transformer/{f}')
os.makedirs('/content/inputs', exist_ok=True)
shutil.copy(f'{DRIVE}/rose_inputs/orbit.mp4', '/content/inputs/orbit.mp4')
shutil.copy('/content/mask_sam2.mp4', '/content/inputs/mask_sam2.mp4')

In [ ]:
%cd /content/ROSE
!python inference.py \
  --validation_videos /content/inputs/orbit.mp4 \
  --validation_masks  /content/inputs/mask_sam2.mp4 \
  --validation_prompts "remove the object" \
  --output_dir /content/rose_out_sam2 --video_length 81 --sample_size 480 832

In [ ]:
# Frames out, visual check, save to Drive (score in the Phase-4 notebook,
# tag 'rose_sam2mask')
import glob
import imageio.v3 as iio
vid = sorted(glob.glob('/content/rose_out_sam2/**/*.mp4', recursive=True))[0]
d = '/content/edited_rose_sam2mask/rgb'
os.makedirs(d, exist_ok=True)
for i, fr in enumerate(iio.imread(vid, plugin='pyav'), start=1):
    Image.fromarray(fr).resize((832, 480)).save(f'{d}/{i:04d}.png')

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].imshow(Image.open('/content/orbit_frames/0041.jpg')); ax[0].set_title('original')
ax[1].imshow(Image.open(f'{d}/0041.png')); ax[1].set_title('ROSE + SAM2 masks')
for a in ax: a.axis('off')
plt.show()

OUT = f'{DRIVE}/checkpoints/phase4_rose'
shutil.copytree('/content/edited_rose_sam2mask', f'{OUT}/edited_rose_sam2mask',
                dirs_exist_ok=True)
print('saved to', OUT)